# Context Breach — Kaggle GRPO Training & Judging Capture

Train the Commander agent and capture every artifact a judge will ask for: training logs, reward/loss curves, leakage / contamination / task-success curves, before/after demo traces, failure cases, and the final checkpoint path.

## Before running

1. **Settings** sidebar: **Accelerator = GPU T4 x2** (or P100), **Internet = On**.
2. **Add Data** → upload `context-breach.zip` as a **private dataset**. Set `DATASET_SLUG` below to match.
3. Run cells top to bottom. Total wall clock: ~45–75 min.

## 1. Stage code into a writable working dir

In [ ]:
import os, shutil, glob

DATASET_SLUG = "context-breach-code"  # set to your Kaggle dataset slug if different
OUTPUT_DIR = "/kaggle/working/outputs/context-breach-grpo"
RESULTS_DIR = "/kaggle/working/context-breach/results"

candidates = [
    f"/kaggle/input/{DATASET_SLUG}/context-breach",
    f"/kaggle/input/{DATASET_SLUG}",
]
src = next((p for p in candidates if os.path.isdir(p) and os.path.exists(os.path.join(p, 'pyproject.toml'))), None)
if src is None:
    print("Could not find pyproject.toml under any candidate path. Available:")
    for p in glob.glob('/kaggle/input/*'):
        print(' ', p)
    raise SystemExit('Set DATASET_SLUG to your dataset slug.')

DST = "/kaggle/working/context-breach"
if os.path.exists(DST):
    shutil.rmtree(DST)
shutil.copytree(src, DST)
os.chdir(DST)
print('cwd:', os.getcwd())
!ls scripts

## 2. Install dependencies

In [ ]:
!pip -q install -e .
!pip -q install -r requirements-training.txt

## 3. Sanity checks

In [ ]:
!python3 -m pytest -q
!openenv validate --verbose || true
!python3 scripts/train_trl_grpo.py --dry-run | tail -25

## 4. Confirm GPU is visible

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
print('device count :', torch.cuda.device_count())
if torch.cuda.is_available():
    print('device name  :', torch.cuda.get_device_name(0))

## 5. Capture the BEFORE baseline (naive vs guarded)

Saves the pre-training comparison plots into `results/`. Judges see what the unsafe baseline does before any training happened.

In [ ]:
!python3 scripts/generate_results.py
!python3 scripts/generate_demo_trace.py > results/demo_trace_before.json
!ls results

## 6. GRPO training

Default config: Qwen2.5-0.5B, 60 episodes, 30 steps, 4 generations, 2048 max completion. ~30–60 min on a T4.

Logs stream live and are also written to `trainer_state.json` inside the output dir for offline plotting.

In [ ]:
!python3 scripts/train_trl_grpo.py \
  --device cuda \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --episodes 60 \
  --max-steps 30 \
  --num-generations 4 \
  --gradient-accumulation-steps 4 \
  --max-completion-length 2048 \
  --output-dir {OUTPUT_DIR} 2>&1 | tee results/training.log

## 7. Plot training curves + capture run metadata

Writes `results/training_reward_curve.png`, `results/training_loss_curve.png`, `results/training_metrics.json`, and `results/run_metadata.json` (model name, steps, batch size, device, final checkpoint path).

In [ ]:
!python3 scripts/plot_training_curves.py --output-dir {OUTPUT_DIR}

## 8. Evaluate the trained model

Rolls the trained checkpoint out on every scenario, captures per-episode leakage / contamination / task-success, and writes:

- `results/trained_eval.json` (full per-episode traces + summary)
- `results/failure_cases.json` (episodes that leaked, overblocked, or failed the task)
- `results/demo_trace_trained.json` (one clean trained-model trace for the demo)

In [ ]:
import os, glob
ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[-1]))
CHECKPOINT = ckpts[-1] if ckpts else OUTPUT_DIR
print('Using checkpoint:', CHECKPOINT)
!python3 scripts/eval_trained_model.py --checkpoint {CHECKPOINT} --episodes 9

## 9. Generate the AFTER plots (3-way comparison)

Naive vs guarded vs trained on reward, leakage, contamination depth, task success.

In [ ]:
!python3 scripts/generate_after_results.py
from IPython.display import Image, display
for name in (
    'training_reward_curve.png',
    'training_loss_curve.png',
    'reward_by_policy_with_trained.png',
    'leakage_by_policy_with_trained.png',
    'contamination_by_policy_with_trained.png',
    'task_success_by_policy_with_trained.png',
):
    p = f'{RESULTS_DIR}/{name}'
    if os.path.exists(p):
        display(Image(p))

## 10. Bundle everything for download

Copies the artifacts judges need into one folder under `/kaggle/working/judging_bundle/` and zips it. After Save Version, the zip is downloadable from the notebook's Output tab.

In [ ]:
import os, shutil, json

BUNDLE = '/kaggle/working/judging_bundle'
if os.path.exists(BUNDLE):
    shutil.rmtree(BUNDLE)
os.makedirs(BUNDLE)

wanted = [
    'run_metadata.json',
    'training.log',
    'training_metrics.json',
    'training_reward_curve.png',
    'training_loss_curve.png',
    'policy_comparison.json',
    'policy_comparison_with_trained.json',
    'reward_by_policy.png',
    'leakage_by_policy.png',
    'contamination_by_policy.png',
    'reward_by_policy_with_trained.png',
    'leakage_by_policy_with_trained.png',
    'contamination_by_policy_with_trained.png',
    'task_success_by_policy_with_trained.png',
    'demo_trace_before.json',
    'demo_trace_trained.json',
    'trained_eval.json',
    'failure_cases.json',
]
for name in wanted:
    src = os.path.join(RESULTS_DIR, name)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(BUNDLE, name))
        print('+', name)
    else:
        print('-', name, '(missing)')

shutil.make_archive('/kaggle/working/judging_bundle', 'zip', BUNDLE)
print('\nBundle:', '/kaggle/working/judging_bundle.zip', '->', os.path.getsize('/kaggle/working/judging_bundle.zip'), 'bytes')

## 11. Optional: push trained weights to Hugging Face Hub

GitHub is wrong for 2 GB checkpoints. Use Hugging Face. Add your token via Kaggle Secrets (Add-ons → Secrets) so it isn't hard-coded.

In [ ]:
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login, HfApi
# login(UserSecretsClient().get_secret('HF_TOKEN'))
# HfApi().upload_folder(
#     folder_path=CHECKPOINT,
#     repo_id='jaswanth28/context-breach-qwen25',
#     repo_type='model',
#     ignore_patterns=['optimizer.pt', 'rng_state.pth', 'scheduler.pt'],
# )